# Example 3: 使用已有的INP来执行

In [2]:
import os
import numpy as np

from ABQflow import BatchAbaqusProcessor, JobSpec, PreparationSpec, HookSpec
from ABQflow import generate_from_inp_files, degenerate_from_array

%reload_ext autoreload
%autoreload 2

ABAQUS_CAE = 'C:/Applications/SIMULIA/Commands/abaqus.bat'
CWD = os.getcwd()

In [ ]:
base_spec = JobSpec(
	job_name = "existing_inp_example",
	workflow = "modular",
	preparation = PreparationSpec(
		kind = "inp_based",
		source_path = "./examples/cae_file/planar_stress_main.inp",
		options = {
			'staging_mode': 'copy',
			'resolve_includes': True
		}
	),
	post_extraction = [
		HookSpec(
			script_path = "./examples/extraction_scripts/get_max_stress_mises.py",
			tasks = [
				{"result_name": "max_stress_mises",},
				{"result_name": "max_displacement",},
			]
		)
	]
)

specs = generate_from_inp_files("./examples/cae_file/scenarios/*.inp", base_spec)

import pprint
pprint.pprint(specs)

[JobSpec(job_name='planar_stress_scenario_1',
         workflow='modular',
         preparation=PreparationSpec(kind='existing_inp',
                                     source_path='c:\\SJTU\\Projects_Code\\24_Abaqus_Pack\\examples\\ExistingInpBatchJob\\cae_file\\scenarios\\planar_stress_scenario_1.inp',
                                     params={},
                                     options={'resolve_includes': True,
                                              'staging_mode': 'copy'}),
         monolithic_script=None,
         monolithic_params={},
         pre_extraction=[],
         post_extraction=[HookSpec(script_path='./examples/ExistingInpBatchJob/cae_file/get_max_stress_mises.py',
                                   tasks=[{'result_name': 'max_stress_mises'},
                                          {'result_name': 'max_displacement'}])],
         meta={'source_inp': 'c:\\SJTU\\Projects_Code\\24_Abaqus_Pack\\examples\\ExistingInpBatchJob\\cae_file\\scenarios\\planar_stre

C:\SJTU\Projects_Code\24_Abaqus_Pack\src\ABQflow\helpers\convert.py:131: UserWarning: Overwriting base_spec.preparation (kind='inp_based') with kind='existing_inp' for file './examples/ExistingInpBatchJob/cae_file/scenarios\planar_stress_scenario_1.inp'
  warnings.warn(
C:\SJTU\Projects_Code\24_Abaqus_Pack\src\ABQflow\helpers\convert.py:131: UserWarning: Overwriting base_spec.preparation (kind='inp_based') with kind='existing_inp' for file './examples/ExistingInpBatchJob/cae_file/scenarios\planar_stress_scenario_2.inp'
  warnings.warn(


In [ ]:
processor = BatchAbaqusProcessor(
	batch_data = specs,
	base_output_dir = os.path.join(CWD, "examples/03_ExistingInpBatchJob/output"),
	cpus_per_job = 4,
	duplicate_mode = "overwrite",
	abaqus_exe = ABAQUS_CAE,
)

In [5]:
outcomes = processor.run_batch(
	num_parallel_jobs = 2,
)

Output()

In [6]:
outcomes

[JobOutcome(job_name='planar_stress_scenario_1', status='COMPLETED', results={'max_stress_mises': 4525.26025390625, 'max_displacement': 4.189039707183838}, error=None),
 JobOutcome(job_name='planar_stress_scenario_2', status='COMPLETED', results={'max_stress_mises': 6787.890625, 'max_displacement': 5.984342098236084}, error=None)]

In [7]:
Y = degenerate_from_array(
	outcomes = outcomes,
	output_names = ["max_stress_mises", "max_displacement"],
)
print(f"output: {Y}")

output: [[4.52526025e+03 4.18903971e+00]
 [6.78789062e+03 5.98434210e+00]]
